In [1]:
import os
import math
import logging
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional, Sequence, Iterator
from contextlib import contextmanager

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

In [4]:
# ✅ valida libs necessárias (evita erro no meio do notebook)
try:
    import pandas as pd
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Falta instalar 'pandas' no seu .venv. Rode: python -m pip install pandas"
    ) from e

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ModuleNotFoundError as e:
    raise RuntimeError(
        "Falta instalar 'pyarrow' no seu .venv. Rode: python -m pip install pyarrow"
    ) from e


# =========================
# LOGGING
# =========================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("bronze_pcsuper")


# =========================
# CONFIGS
# =========================
@dataclass(frozen=True)
class PostgresConfig:
    host: str
    port: int
    dbname: str
    user: str
    password: str
    schema: str
    connect_timeout: int = 10


@dataclass(frozen=True)
class BronzeConfig:
    base_path: Path
    chunk_rows: int = 200_000


@dataclass(frozen=True)
class TableSpec:
    schema: str
    table: str
    columns: Optional[Sequence[str]] = None
    where: Optional[str] = None


def load_configs() -> tuple[PostgresConfig, BronzeConfig]:
    load_dotenv()

    host = os.getenv("POSTGRES_DW_HOST", "").strip()
    port = int(os.getenv("POSTGRES_DW_PORT", "5433"))
    dbname = os.getenv("POSTGRES_DW_DB", "").strip()
    user = os.getenv("POSTGRES_DW_USER", "").strip()
    password = os.getenv("POSTGRES_DW_PASSWORD", "").strip()
    schema = os.getenv("POSTGRES_DW_SCHEMA", "public").strip() or "public"

    bronze_base = os.getenv("BRONZE_BASE_PATH", "./datalake/bronze").strip()
    chunk_rows = int(os.getenv("BRONZE_CHUNK_ROWS", "200000"))

    if not all([host, dbname, user, password]):
        raise RuntimeError("Variáveis Postgres incompletas no .env (HOST/DB/USER/PASSWORD).")

    pg = PostgresConfig(
        host=host,
        port=port,
        dbname=dbname,
        user=user,
        password=password,
        schema=schema,
    )
    bronze = BronzeConfig(base_path=Path(bronze_base), chunk_rows=chunk_rows)
    return pg, bronze


def pg_conn_str(cfg: PostgresConfig) -> str:
    return (
        f"host={cfg.host} port={cfg.port} dbname={cfg.dbname} "
        f"user={cfg.user} password={cfg.password} connect_timeout={cfg.connect_timeout}"
    )


@contextmanager
def pg_connection(cfg: PostgresConfig) -> Iterator[psycopg.Connection]:
    conn: Optional[psycopg.Connection] = None
    try:
        logger.info("Conectando ao Postgres %s:%s/%s", cfg.host, cfg.port, cfg.dbname)
        conn = psycopg.connect(pg_conn_str(cfg), row_factory=dict_row)

        # ✅ server-side cursor precisa de transação ativa
        conn.autocommit = False

        yield conn

        # leitura apenas; mas mantém "limpo" caso algo abra transação
        conn.commit()

    except Exception:
        if conn:
            conn.rollback()
        raise

    finally:
        if conn:
            conn.close()


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def build_select_sql(spec: TableSpec) -> str:
    cols = ", ".join([f'"{c}"' for c in spec.columns]) if spec.columns else "*"
    sql = f'SELECT {cols} FROM "{spec.schema}"."{spec.table}"'
    if spec.where:
        sql += f" WHERE {spec.where}"
    return sql


def count_rows(conn: psycopg.Connection, spec: TableSpec) -> int:
    sql = f'SELECT COUNT(*) AS n FROM "{spec.schema}"."{spec.table}"'
    if spec.where:
        sql += f" WHERE {spec.where}"
    with conn.cursor(row_factory=dict_row) as cur:
        cur.execute(sql)
        return int(cur.fetchone()["n"])


def write_parquet_chunk(
    df: pd.DataFrame,
    out_dir: Path,
    chunk_idx: int,
    ingestion_dt: datetime
) -> Path:
    df["_ingestion_ts_utc"] = ingestion_dt.isoformat()
    df["_ingestion_date"] = ingestion_dt.date().isoformat()

    ensure_dir(out_dir)
    file_path = out_dir / f"part-{chunk_idx:05d}.parquet"

    table = pa.Table.from_pandas(df, preserve_index=False)
    pq.write_table(table, file_path, compression="snappy")
    return file_path


def extract_to_bronze(
    conn: psycopg.Connection,
    bronze: BronzeConfig,
    spec: TableSpec,
    ingestion_dt: datetime
) -> list[Path]:

    total = count_rows(conn, spec)
    if total == 0:
        logger.warning("Tabela vazia: %s.%s", spec.schema, spec.table)
        return []

    chunks = max(1, math.ceil(total / bronze.chunk_rows))
    logger.info(
        "Extraindo %s.%s | linhas=%s | chunk_rows=%s | chunks=%s",
        spec.schema, spec.table, total, bronze.chunk_rows, chunks
    )

    ingest_date = ingestion_dt.date().isoformat()
    out_dir = bronze.base_path / spec.schema / spec.table / f"ingest_date={ingest_date}"

    sql = build_select_sql(spec)
    written: list[Path] = []

    # ✅ cursor nomeado dentro de transação
    # ✅ nome fixo/limpo evita problema com caracteres do schema/tabela
    with conn.transaction():
        with conn.cursor(name="cur_bronze_stream", row_factory=dict_row) as cur:
            cur.itersize = bronze.chunk_rows
            cur.execute(sql)

            chunk_idx = 0
            while True:
                rows = cur.fetchmany(bronze.chunk_rows)
                if not rows:
                    break

                df = pd.DataFrame(rows)
                path = write_parquet_chunk(df, out_dir, chunk_idx, ingestion_dt)
                written.append(path)
                logger.info("Gravado: %s | linhas=%s", path, len(df))
                chunk_idx += 1

    logger.info("Concluído %s.%s | arquivos=%s", spec.schema, spec.table, len(written))
    return written


def main() -> None:
    pg_cfg, bronze_cfg = load_configs()
    ingestion_dt = datetime.now(timezone.utc)

    spec = TableSpec(schema=pg_cfg.schema, table="pcsuper")
    logger.info("Bronze base path: %s", bronze_cfg.base_path)

    with pg_connection(pg_cfg) as conn:
        extract_to_bronze(conn, bronze_cfg, spec, ingestion_dt)


if __name__ == "__main__":
    main()

2026-02-07 23:47:22,563 | INFO | bronze_pcsuper | Bronze base path: /home/Projetos/datalake/bronze
2026-02-07 23:47:22,564 | INFO | bronze_pcsuper | Conectando ao Postgres 209.50.228.232:5433/ERP


2026-02-07 23:47:23,473 | INFO | bronze_pcsuper | Extraindo public.pcsuper | linhas=47 | chunk_rows=200000 | chunks=1
2026-02-07 23:47:23,992 | INFO | bronze_pcsuper | Gravado: /home/Projetos/datalake/bronze/public/pcsuper/ingest_date=2026-02-07/part-00000.parquet | linhas=47
2026-02-07 23:47:24,377 | INFO | bronze_pcsuper | Concluído public.pcsuper | arquivos=1
